# Bank marketing: will the client subscribe?

A typical first-pass notebook. It runs, the score looks great, and it contains several methodology mistakes. Review it with:

```bash
python -m dclab_rnd.copilot review dclab_rnd/copilot/examples/leaky_bank_marketing.ipynb
```

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.utils import resample

In [ ]:
ROOT = Path.cwd()
while not (ROOT / "data/public").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
X = pd.read_parquet(ROOT / "data/public/bank_marketing/X.parquet")
y = pd.read_parquet(ROOT / "data/public/bank_marketing/y.parquet")["target"]
df = X.assign(target=y)
df.head()

In [ ]:
# call length is the strongest predictor, keep it
features = ["age", "balance", "duration", "campaign", "pdays", "previous", "job", "month"]
df["job_rate"] = df.groupby("job")["target"].transform("mean")
features.append("job_rate")

In [ ]:
# balance the classes so the model sees enough subscribers
minority = df[df.target == 1]
majority = df[df.target == 0]
df_bal = pd.concat([majority, resample(minority, n_samples=len(majority))])

In [ ]:
scaler = StandardScaler()
X_all = scaler.fit_transform(df_bal[features])
X_train, X_test, y_train, y_test = train_test_split(X_all, df_bal["target"], test_size=0.2)

In [ ]:
best_acc, best_depth = 0, None
for depth in [4, 8, 16, None]:
    model = RandomForestClassifier(n_estimators=200, max_depth=depth)
    model.fit(X_train, y_train)
    acc = accuracy_score(y_test, model.predict(X_test))
    if acc > best_acc:
        best_acc, best_depth = acc, depth
print(best_depth, best_acc)